# AI4SkIN multi-site fairness — golden replication

Runs `fmm-fairness evaluate` on the published HUSC + HCUV confusion
matrices and asserts the headline weighted F1 gap reproduces Table 6
of Pereiro 2026 (TFG, UPV).

Expected: weighted F1 gap = 0.1657 ± 0.005, permutation p < 0.05.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

HERE = Path.cwd()
if HERE.name == "ai4skin-replication":
    EXAMPLE_DIR = HERE
    REPO_ROOT = HERE.parents[1]
else:
    REPO_ROOT = HERE
    EXAMPLE_DIR = REPO_ROOT / "examples" / "ai4skin-replication"

print("repo root:    ", REPO_ROOT)
print("example dir:  ", EXAMPLE_DIR)

In [ ]:
# 1. Regenerate the CSVs deterministically (no-op if seed matches the commit).
subprocess.run(
    [sys.executable, str(EXAMPLE_DIR / "build_dataset.py")],
    check=True,
)

In [ ]:
# 2. Run the CLI.
out_dir = EXAMPLE_DIR / "out"
subprocess.run(
    [
        sys.executable, "-m", "fmm_fairness.cli", "evaluate",
        str(EXAMPLE_DIR / "predictions.csv"),
        "--protected-attrs", "site",
        "--site-attribute", "site",
        "--rater-cols", "doc1,doc2,doc3,doc4,doc5,doc6,doc7,doc8,doc9,doc10",
        "--bootstrap-method", "bca",
        "--bootstrap-iters", "2000",
        "--permutation-iters", "2000",
        "--output", str(out_dir),
    ],
    check=True,
    cwd=REPO_ROOT,
)

In [ ]:
# 3. Load the evidence pack and surface headline numbers.
evidence = json.loads((out_dir / "fairness-evidence.json").read_text(encoding="utf-8"))
wf1 = evidence["per_attribute_metrics"]["site"]["weighted_f1_gap"]
macro = evidence["per_attribute_metrics"]["site"]["macro_f1_gap"]
kappa = evidence["inter_rater_agreement"]["ai_vs_pooled_raters_kappa"]

per_group = {g["group"]: (g["n"], round(g["value"], 4)) for g in wf1["per_group"]}
print("Weighted F1 by site:")
for site_name, (n, value) in per_group.items():
    print(f"  {site_name:<6}  n={n:<4}  F1 = {value}")
print()
print(f"Inter-site weighted F1 gap   : {wf1['gap']:.4f}")
print(f"  BCa CI95                   : [{wf1['gap_ci_low']:.4f}, {wf1['gap_ci_high']:.4f}]")
print(f"  Bootstrap SE               : {wf1['bootstrap_se']:.4f}")
print(f"  Permutation p (H0: no gap) : {wf1['permutation_p_value']:.4f}")
print(f"  MDE @ 80% power            : {wf1['minimum_detectable_effect']:.4f}")
print()
print(f"Inter-site macro F1 gap      : {macro['gap']:.4f}")
print(f"AI vs pooled-raters Cohen κ  : {kappa['value']:.4f}  (CI95 [{kappa['ci_low']:.4f}, {kappa['ci_high']:.4f}])")

In [ ]:
# 4. Acceptance gate — fails the notebook (and CI) if the numbers drift.
TARGET_GAP = 0.1657
TOL = 0.005
assert abs(wf1["gap"] - TARGET_GAP) <= TOL, (
    f"weighted F1 gap {wf1['gap']:.4f} drifted from TFG target {TARGET_GAP} ± {TOL}"
)
assert wf1["permutation_p_value"] is not None and wf1["permutation_p_value"] < 0.05, (
    f"permutation p {wf1['permutation_p_value']} should be < 0.05 for the TFG headline gap"
)
assert 0.70 < kappa["value"] < 0.90, (
    f"AI-vs-pooled κ {kappa['value']:.4f} outside the published-plausible 0.70-0.90 band"
)
print("OK: replication numbers within tolerance of TFG Table 6.")